# AI-sfa 이펙트 LoRA 2차 — 컬러 이펙트 / 임팩트 프레임 분리 (Kaggle T4)
1차(한 LoRA)는 자료 40%가 저채도라 색을 지시해도 회색으로 그렸다. 사용자 결정(2026-10-25): 나카무라의 화려한 색감을 지키려고 둘로 나눈다.
- **컬러 이펙트 LoRA** `nkmr_fx_color`: 색 있는 그림 597장(2번씩) + 옅은 그림 206장(1번, `muted colors` 태그). 4바퀴, 바퀴마다 저장.
- **임팩트 프레임 LoRA** `nkmr_impact`: 흑백·반전·임팩트 229장, 신호 낱말 `nkmr_impact`. 6바퀴, 2바퀴마다 저장.
- 설정: **GPU T4 x2**, **Internet On** → **Save Version → Save & Run All (Commit)**. 약 6시간 예상(12시간 한도 안).
- 끝나면 Output의 `lora_v2_result.zip`(LoRA + 확인 그림)을 Drive 공유 링크로 알려 준다. 자료 분리: 저장소 tools/lora_split.py

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2
!free -g | head -2

In [ ]:
# 1) 학습 환경: 서버에서 불러오기까지 확인한 조합을 별도 환경(/tmp/env)에 그대로 깐다(Kaggle 기본 패키지와 섞이지 않게)
!pip -q install uv
!uv venv -q /tmp/env --python 3.10
!uv pip install -q --python /tmp/env/bin/python torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu124
!git clone -q --depth 1 --branch v0.9.1 https://github.com/kohya-ss/sd-scripts.git /tmp/sd-scripts
!cd /tmp/sd-scripts && uv pip install -q --python /tmp/env/bin/python -r requirements.txt
!uv pip install -q --python /tmp/env/bin/python lycoris-lora==3.1.1 "numpy==1.26.4"
!cd /tmp/sd-scripts && /tmp/env/bin/python -c "import torch, library, lycoris, bitsandbytes, cv2; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 2) 학습 자료 2차(Drive 공유 링크)와 기본 모델(Hugging Face)
!curl -sSL -o /tmp/data.zip "https://drive.usercontent.google.com/download?id=1TIGJDAtj4sndB_GH3Dg7ifOAlSiYmjyF&export=download&confirm=t"
!mkdir -p /tmp/data && cd /tmp/data && unzip -q -o /tmp/data.zip && for d in color color_muted impact; do echo $d $(ls $d/*.jpg | wc -l); done
!curl -sSL -C - -o /tmp/ilxl.safetensors https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors
!ls -la /tmp/ilxl.safetensors

In [ ]:
# 3) 자료 설정 두 개(캡션 섞기 끔: 텍스트 인코더 출력을 미리 계산하므로)
import os, glob
T = '''[general]
caption_extension = ".txt"
shuffle_caption = false

[[datasets]]
resolution = 1024
batch_size = 1
enable_bucket = true
min_bucket_reso = 640
max_bucket_reso = 1536
bucket_reso_steps = 64
'''
open('/tmp/color.toml','w').write(T + '''
  [[datasets.subsets]]
  image_dir = "/tmp/data/color"
  num_repeats = 2

  [[datasets.subsets]]
  image_dir = "/tmp/data/color_muted"
  num_repeats = 1
''')
open('/tmp/impact.toml','w').write(T + '''
  [[datasets.subsets]]
  image_dir = "/tmp/data/impact"
  num_repeats = 1
''')
print(open('/tmp/color.toml').read())

In [ ]:
# 4) 학습: 컬러 먼저(약 4시간), 임팩트 다음(약 1시간). T4: 기본 모델 fp8 + fp16 계산, VAE fp32
!mkdir -p /kaggle/working/out
def train(name, toml, epochs, every):
    cmd = f"""cd /tmp/sd-scripts && /tmp/env/bin/python sdxl_train_network.py \
  --pretrained_model_name_or_path /tmp/ilxl.safetensors --dataset_config {toml} \
  --output_dir /kaggle/working/out --output_name {name} \
  --network_module lycoris.kohya --network_dim 32 --network_alpha 8 \
  --network_args algo=locon preset=full conv_dim=16 conv_alpha=4 \
  --network_train_unet_only --unet_lr 4e-4 \
  --optimizer_type AdamW8bit --lr_scheduler cosine --lr_warmup_steps 100 \
  --max_train_epochs {epochs} --save_every_n_epochs {every} --save_model_as safetensors \
  --mixed_precision fp16 --save_precision fp16 --fp8_base --no_half_vae --sdpa --gradient_checkpointing \
  --cache_latents --cache_latents_to_disk --cache_text_encoder_outputs --cache_text_encoder_outputs_to_disk \
  --min_snr_gamma 5 --max_data_loader_n_workers 2 --seed 42"""
    r = os.system(cmd)
    print(name, 'exit', r)
train('nkmr_fx_color', '/tmp/color.toml', 4, 1)
train('nkmr_impact', '/tmp/impact.toml', 6, 2)
!ls -la /kaggle/working/out

In [ ]:
# 5) 확인 그림: 같은 글·같은 시드로 기본 모델 / 각 바퀴 (LyCORIS는 --network_merge로 합쳐 그림)
NEG = " --n lowres, worst quality, text, watermark --w 1344 --h 768 --s 28 --l 6 --d 1234"
color_p = [
 "nkmr_fx, anime screencap, explosion, smoke, billowing mass, white core orange rim, layered cel shading, no outline, sharp tips, expanding, large effect, no humans, simple background, wide shot",
 "nkmr_fx, anime screencap, lightning, embers, streaks, particles, white core blue rim, no outline, sharp tips, rapid expansion, no humans, city, wide shot",
 "nkmr_fx, anime screencap, flash, light bloom, glowing aura, white core blue rim, layered cel shading, peak, large effect, no humans, space",
 "nkmr_fx, anime screencap, debris, dust cloud, billowing mass, flat color, simple cel shading, dispersing, flying, no humans, rocky ground, wide shot",
 "nkmr_fx, anime screencap, energy beam, shockwave, ring, white core pink rim, layered cel shading, sustained, large effect, no humans, sky",
 "nkmr_fx, anime screencap, fire, smoke, pillar, white core orange rim, layered cel shading, rough brush strokes, expanding, no humans, city, from below",
]
impact_p = [
 "nkmr_impact, anime screencap, impact frame, flash, radial, monochrome, rough brush strokes, peak, full screen effect, no humans, abstract background",
 "nkmr_impact, anime screencap, impact frame, explosion, speed lines, monochrome, inverted colors, sharp tips, effect onset, full screen effect, silhouette",
 "nkmr_impact, anime screencap, impact frame, lightning, shockwave, ring, monochrome, no outline, peak, large effect, no humans, city",
]
def sample(tag, prompts, ckpts):
    open(f'/tmp/p_{tag}.txt','w').write("\n".join(p + NEG for p in prompts))
    for name, c in [("base", None)] + [(os.path.basename(c)[:-12], c) for c in ckpts]:
        out = f"/kaggle/working/samples_{tag}/{name}"
        os.makedirs(out, exist_ok=True)
        net = f"--network_module lycoris.kohya --network_weights {c} --network_mul 1.0 --network_merge" if c else ""
        os.system(f"cd /tmp/sd-scripts && /tmp/env/bin/python sdxl_gen_img.py --ckpt /tmp/ilxl.safetensors --outdir {out} --from_file /tmp/p_{tag}.txt --images_per_prompt 1 --fp16 --no_half_vae --sdpa --sampler euler_a {net} > {out}/log.txt 2>&1")
        print(tag, name, len(glob.glob(out + '/*.png')), 'images')
sample("color", color_p, sorted(glob.glob('/kaggle/working/out/nkmr_fx_color*.safetensors')))
sample("impact", impact_p, sorted(glob.glob('/kaggle/working/out/nkmr_impact*.safetensors')))

In [ ]:
# 6) 격자(줄 = 기본/바퀴, 칸 = 글) + 묶기
from PIL import Image, ImageDraw
def grid(tag):
    rows = sorted(glob.glob(f'/kaggle/working/samples_{tag}/*/'), key=lambda r: (r.rstrip('/').split('/')[-1] != 'base', r))
    rows = [(r.rstrip('/').split('/')[-1], sorted(glob.glob(r + '*.png'))) for r in rows]
    rows = [(n, ims) for n, ims in rows if ims]
    if not rows: return
    W = Image.new('RGB', (448 * max(len(i) for _, i in rows), 256 * len(rows)), 'black')
    d = ImageDraw.Draw(W)
    for y, (n, ims) in enumerate(rows):
        for x, p in enumerate(ims):
            W.paste(Image.open(p).resize((448, 256)), (448 * x, 256 * y))
        d.rectangle((0, 256 * y, 190, 256 * y + 20), fill='black'); d.text((4, 256 * y + 4), n, fill='white')
    W.save(f'/kaggle/working/grid_{tag}.jpg', quality=88)
    print(tag, [(n, len(i)) for n, i in rows])
grid("color"); grid("impact")
!cd /kaggle/working && zip -q -r lora_v2_result.zip out samples_color samples_impact grid_color.jpg grid_impact.jpg && ls -la /kaggle/working